# PIE paper-weights reproduction (Colab)

Loads the published Keras `.h5` checkpoints into our PyTorch port and runs **evaluation** on the dataset you already have on Drive (set05 val split). No training happens here.

Preconditions:
- You've already run the Colab checklist (`docs/colab_checklist.md`).
- `$PIE_PATH` holds set05 MP4s, extracted frames, and annotations.

This notebook covers:
1. Pull the latest code.
2. Install deps (no-ops if already installed).
3. Convert each of the three paper `.h5` checkpoints to `.safetensors`.
4. Run `pie-eval --keras-h5 ...` for intent / trajectory / speed on set05.

Heads up: the paper reports its test metrics on **set03**. We don't have set03 on Drive (~25 GB), so these numbers will differ from paper's reported values — but they prove the full loading + eval pipeline is wired right.

## 1. Mount Drive + set `PIE_PATH`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.environ['PIE_PATH'] = '/content/drive/MyDrive/Portofolios/Protofolio_2026/pedestrian_estimation_pie/pie_data'
assert os.path.isdir(os.environ['PIE_PATH']), f"PIE_PATH missing: {os.environ['PIE_PATH']}"
!ls "$PIE_PATH"

## 2. Pull the latest code + install deps

In [ ]:
%cd /content
![ -d PIE_Pedestrian_Estimation ] || git clone -b claude/tensorflow-to-pytorch-conversion-0SlwI https://github.com/venetisgr/PIE_Pedestrian_Estimation.git
%cd /content/PIE_Pedestrian_Estimation
!git checkout claude/tensorflow-to-pytorch-conversion-0SlwI
!git pull
!git log --oneline -3

In [ ]:
!pip install -q -r requirements.txt

## 3. Sanity: tests pass on this runtime

In [ ]:
!python -m pytest tests/test_keras_to_torch.py -q

## 4. Convert the three `.h5` checkpoints to `.safetensors`

One-shot. Writes next to the `.h5` files. Subsequent runs can skip this cell.

In [ ]:
!python -m pie_pytorch.cli.convert \
  --task intent \
  --h5 data/pie/intention/context_loc_pretrained/model.h5 \
  --out data/pie/intention/context_loc_pretrained/model.safetensors

In [ ]:
!python -m pie_pytorch.cli.convert \
  --task trajectory \
  --h5 data/pie/trajectory/loc_intent_speed_pretrained/model.h5 \
  --out data/pie/trajectory/loc_intent_speed_pretrained/model.safetensors

In [ ]:
!python -m pie_pytorch.cli.convert \
  --task speed \
  --h5 data/pie/speed/speed_pretrained/model.h5 \
  --out data/pie/speed/speed_pretrained/model.safetensors

## 5. Evaluate paper weights on set05

### 5a. Intent (expected: binary cross-entropy, accuracy, F1)

In [ ]:
!python -m pie_pytorch.cli.eval \
  --config pie_pytorch/configs/intent_colab.yaml \
  --keras-h5 data/pie/intention/context_loc_pretrained/model.h5

### 5b. Trajectory (expected: MSE, C-MSE)

In [ ]:
!python -m pie_pytorch.cli.eval \
  --config pie_pytorch/configs/trajectory_colab.yaml \
  --keras-h5 data/pie/trajectory/loc_intent_speed_pretrained/model.h5

### 5c. Speed (expected: MSE)

**Note**: the paper trained speed with a 1-dim zero-filled decoder input; our default `speed_colab.yaml` uses `dec_feature_size: 0` for scratch training. For paper-weight eval we override to `dec_feature_size=1` via `--override`.

In [ ]:
!python -m pie_pytorch.cli.eval \
  --config pie_pytorch/configs/speed_colab.yaml \
  --keras-h5 data/pie/speed/speed_pretrained/model.h5

## Done

Those are the paper's trained weights running through our PyTorch pipeline. Paste the final metric lines back to the thread so we can cross-check the numbers look sensible.